# Coulomb collisions

Charged markers in the same grid cell scatter off each other through binary Coulomb
collisions: `plasmacoll.coulomb` pairs them at random in every cell (Takizuka and Abe 1977) and turns
the relative velocity of each pair by an angle drawn from Nanbu's (1997) cumulative
small-angle distribution. Each pair keeps its momentum and energy. This notebook checks
two rates of the NRL Plasma Formulary with the collision operator alone: the temperature
relaxation of two ion species and the isotropisation of a bi-Maxwellian. The markers sit
in a handful of cells of unit volume; no field is solved, which is the zero-dimensional
limit of the operator a particle-in-cell code applies after every push.

You need the idea of a marker weight (physical particles per marker) and of a
temperature in eV.

## The scattering parameter

A pair of masses $m_a$, $m_b$, charges $q_a$, $q_b$ and relative speed $u$ turns over
one collision step $\Delta t$ by an angle whose mean $\langle 1-\cos\chi\rangle =
1 - e^{-s}$ follows from

$$s = \frac{q_a^2 q_b^2 \ln\Lambda\, n\, \Delta t}{4\pi\varepsilon_0^2\mu^2 u^3},
\qquad \mu = \frac{m_a m_b}{m_a + m_b},$$

with $n$ the density of the partner species (for unequal marker weights, the weighted
form of Pérez et al. 2012). Slow pairs have a large $s$: their angle saturates, so the
energy they exchange is underestimated unless $s$ at the thermal speed is small. Keep it
below about $3\times10^{-3}$ for rates within a few per cent.

In [ ]:
import plasma_plots as pp

pp.set_backend("plotly")  # interactive figures, embedded in the docs
pp.set_math("latex")  # formulas in the figures typeset as LaTeX

In [ ]:
import numpy as np
import xarray as xr

from plasmacoll.constants import ELEMENTARY_CHARGE as e
from plasmacoll.constants import PROTON_MASS as mp
from plasmacoll.constants import VACUUM_PERMITTIVITY as eps0
from plasmacoll.coulomb import (
    CellMarkers,
    collide_inter,
    collide_intra,
    pair_coefficient,
    relaxation_rate,
)

rng = np.random.default_rng(1)
density, log_lambda, cells = 1e20, 10.0, 30  # m^-3, Coulomb logarithm, cells of 1 m^3


def maxwellian(mass, temperature, count):
    # (3, count) velocities with exactly no drift and the temperature (eV)
    v = rng.normal(0.0, 1.0, (3, count))
    v -= v.mean(axis=1, keepdims=True)
    return v * np.sqrt(3 * temperature * e / mass / np.mean(np.sum(v**2, axis=0)))


def in_cells(velocities, mass, charge):
    # the markers in `cells` cells of equal size, with the physical density
    count = velocities.shape[1]
    keys = np.repeat(np.arange(cells), count // cells)
    weights = np.full(count, density * cells / count)
    return CellMarkers.from_sorted(velocities, weights, keys, cells, mass, charge)


def temperature(velocities, mass):
    return mass * np.mean(np.sum(velocities**2, axis=0)) / (3 * e)

## Temperature relaxation

Protons at 2 eV and ions of four proton masses at 1 eV exchange energy at the rate
$dT_a/dt = \bar\nu_{ab}(T_b - T_a)$ with

$$\bar\nu_{ab} = \frac{\sqrt{2}\, n_b q_a^2 q_b^2 \ln\Lambda \sqrt{m_a m_b}}
{6\pi^{3/2}\varepsilon_0^2 (m_a T_b + m_b T_a)^{3/2}}$$

(`relaxation_rate`). The rate assumes that both species stay Maxwellian; here the
collisions of each species with itself are made a hundred times faster (a larger
Coulomb logarithm) to keep them so, which tests the collisions between the species.
The theory curve integrates the two rate equations alongside.

In [ ]:
count, steps = 60_000, 300
ma, mb, ta, tb = mp, 4 * mp, 2.0, 1.0
va, vb = maxwellian(ma, ta, count), maxwellian(mb, tb, count)
nu0 = relaxation_rate(e, ma, ta, e, mb, density, tb, log_lambda) + relaxation_rate(
    e, mb, tb, e, ma, density, ta, log_lambda
)
dt = 0.7 / nu0 / steps
thermal = np.sqrt(3 * (ta * e / ma + tb * e / mb))
s_thermal = pair_coefficient(e, e, ma, mb, log_lambda, dt) * density / thermal**3
print(f"s at the thermal speed: {s_thermal:.1e}")

history, theory = [], []
theory_a, theory_b = ta, tb
for _ in range(steps):
    history.append((temperature(va, ma), temperature(vb, mb)))
    theory.append((theory_a, theory_b))
    ra = relaxation_rate(e, ma, theory_a, e, mb, density, theory_b, log_lambda)
    rb = relaxation_rate(e, mb, theory_b, e, ma, density, theory_a, log_lambda)
    theory_a, theory_b = (
        theory_a + dt * ra * (theory_b - theory_a),
        theory_b + dt * rb * (theory_a - theory_b),
    )
    a, b = in_cells(va, ma, e), in_cells(vb, mb, e)
    collide_intra(a, dt, 1.0, 100 * log_lambda, rng=rng)
    collide_intra(b, dt, 1.0, 100 * log_lambda, rng=rng)
    collide_inter(a, b, dt, 1.0, log_lambda, rng=rng)

time = ("t", nu0 * dt * np.arange(steps), {"long_name": r"$\nu_0 t$"})
relaxation = xr.Dataset(
    {
        "T_protons": ("t", [h[0] for h in history], {"units": "eV"}),
        "T_ions": ("t", [h[1] for h in history], {"units": "eV"}),
        "difference": ("t", [h[0] - h[1] for h in history], {"units": "eV"}),
        "theory": ("t", [t[0] - t[1] for t in theory], {"units": "eV"}),
    },
    coords={"t": time},
)
relaxation.difference.attrs["long_name"] = r"$T_\mathrm{p} - T_\mathrm{i}$"
measured = np.log(relaxation.difference[-1] / relaxation.difference[0])
expected = np.log(relaxation.theory[-1] / relaxation.theory[0])
print(f"relaxation rate / NRL rate: {float(measured / expected):.3f}")
relaxation.difference.plasma.plot.timeseries(
    reference={"NRL": (relaxation.t.values, relaxation.theory.values)}
);

The measured rate is within a few per cent of the NRL rate, a little below it: the
slowest pairs saturate. Halving the step (doubling `steps`) brings it closer to one.
Total energy is kept exactly, so the two temperatures meet at their mean.

## Isotropisation of a bi-Maxwellian

Protons with $T_\perp = 2T_\parallel$ relax toward one temperature through their
collisions with each other, at the NRL rate

$$\frac{dT_\perp}{dt} = -\frac12\frac{dT_\parallel}{dt} = -\nu_T (T_\perp - T_\parallel),
\quad \nu_T = \frac{q^4 n \ln\Lambda}{8\pi^{3/2}\varepsilon_0^2 m^{1/2} T_\parallel^{3/2}}
\frac{-3 + (A+3)\arctan\sqrt{A}/\sqrt{A}}{A^2},\quad A = \frac{T_\perp}{T_\parallel} - 1.$$

The formula holds for a bi-Maxwellian, which the distribution stays only at first; the
comparison is over the first tenth of the relaxation time.

In [ ]:
def isotropisation_rate(t_par, t_perp):
    anisotropy = t_perp / t_par - 1
    factor = -3 + (anisotropy + 3) * np.arctan(np.sqrt(anisotropy)) / np.sqrt(
        anisotropy
    )
    return (
        e**4
        * density
        * log_lambda
        * factor
        / anisotropy**2
        / (8 * np.pi**1.5 * eps0**2 * np.sqrt(mp) * (t_par * e) ** 1.5)
    )


count, steps = 300_000, 100
v = np.concatenate([maxwellian(mp, 2.0, count)[:2], maxwellian(mp, 1.0, count)[2:]])
nu_t = isotropisation_rate(1.0, 2.0)
dt = 0.1 / nu_t / steps
rows, theory_par, theory_perp = [], 1.0, 2.0
for _ in range(steps):
    rows.append(
        (
            mp * np.mean(v[:2] ** 2) / e,
            mp * np.mean(v[2] ** 2) / e,
            theory_perp - theory_par,
        )
    )
    change = (
        dt * isotropisation_rate(theory_par, theory_perp) * (theory_perp - theory_par)
    )
    theory_par, theory_perp = theory_par + 2 * change, theory_perp - change
    collide_intra(in_cells(v, mp, e), dt, 1.0, log_lambda, rng=rng)

rows = np.array(rows)
isotropy = xr.Dataset(
    {
        "anisotropy": (
            "t",
            rows[:, 0] - rows[:, 1],
            {"units": "eV", "long_name": r"$T_\perp - T_\parallel$"},
        ),
        "theory": ("t", rows[:, 2], {"units": "eV"}),
    },
    coords={"t": ("t", nu_t * dt * np.arange(steps), {"long_name": r"$\nu_T t$"})},
)
rate = np.log(isotropy.anisotropy[-1] / isotropy.anisotropy[0]) / np.log(
    isotropy.theory[-1] / isotropy.theory[0]
)
print(f"isotropisation rate / NRL rate: {float(rate):.3f}")
isotropy.anisotropy.plasma.plot.timeseries(
    reference={"NRL": (isotropy.t.values, isotropy.theory.values)}
);

## In a particle-in-cell code

A code sorts the markers of each species by cell (its own cell keys, e.g. from the
positions and the grid) and hands views of the sorted velocity and weight arrays to
`CellMarkers.from_sorted`, so the collisions write the markers in place. Without
`coulomb_log`, the Coulomb logarithm comes from the NRL formulas with the density and
temperature of every cell (`coulomb_logarithm`). When several MPI ranks hold markers of
the same cells, `reduce` (an in-place sum over the ranks, e.g. `comm.Allreduce`) makes
the densities, temperatures and rates those of the whole plasma. On CuPy arrays the
CUDA versions of the kernels run, one thread per cell.

## Try it

Set `steps = 100` in the relaxation: $s$ triples and the measured rate drops by several
per cent. Then use the same Coulomb logarithm for the collisions of each species with
itself as between them: the species no longer stay Maxwellian, and the relaxation is
slower than the NRL rate, which assumes Maxwellians.